# Comparación: ensamblaje, red neuronal y LLM

Mismo problema que `modelos_proyecto.ipynb`: predecir `UNITS`, con la preparación y la división por fecha de `pipeline_proyecto.py`. Métrica: RMSE. Comparo el mejor ensamblaje de la sesión anterior (bosque con `min_samples_leaf=20`), una red neuronal simple y un LLM usado con prompt.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

from pipeline_proyecto import cargar_y_dividir, construir_preparacion

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

Entrenamiento: (159871, 11) Prueba: (39962, 11)


## Los dos modelos

In [2]:
resultados = {}
entrenados = {}


def evaluar(nombre, modelo):
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    pipeline.fit(X_ent, y_ent)
    segundos = time.perf_counter() - inicio
    rmse_ent = root_mean_squared_error(y_ent, pipeline.predict(X_ent))
    rmse_prueba = root_mean_squared_error(y_prueba, pipeline.predict(X_prueba))
    resultados[nombre] = {
        "modelo": nombre,
        "rmse_entrenamiento": round(rmse_ent, 4),
        "rmse_prueba": round(rmse_prueba, 4),
        "brecha": round(rmse_prueba - rmse_ent, 4),
        "segundos": round(segundos, 3),
    }
    entrenados[nombre] = pipeline
    print(f"{nombre:28s} entrenamiento {rmse_ent:7.4f}   prueba {rmse_prueba:7.4f}   {segundos:6.2f} s")


evaluar("bosque min_samples_leaf=20", RandomForestRegressor(
    n_estimators=100, min_samples_leaf=20, random_state=42, n_jobs=-1))

bosque min_samples_leaf=20   entrenamiento  2.8233   prueba  2.9332    26.10 s


In [3]:
# red pequena: dos capas ocultas; se detiene sola si la validacion deja de mejorar
red = MLPRegressor(hidden_layer_sizes=(32, 16), early_stopping=True, max_iter=200, random_state=42)
evaluar("red neuronal (32, 16)", red)

mlp = entrenados["red neuronal (32, 16)"].named_steps["modelo"]
parametros = sum(c.size for c in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print("Parametros de la red:", parametros, "| iteraciones:", mlp.n_iter_)
resultados["red neuronal (32, 16)"]["parametros"] = parametros

red neuronal (32, 16)        entrenamiento  2.8186   prueba  2.9572    20.57 s
Parametros de la red: 1409 | iteraciones: 77


In [4]:
tabla = pd.DataFrame(resultados.values())
tabla

,modelo,rmse_entrenamiento,rmse_prueba,brecha,segundos,parametros
0,bosque min_samples_leaf=20,2.8233,2.9332,0.1099,26.097,NaN
1,"red neuronal (32, 16)",2.8186,2.9572,0.1386,20.566,1409.0


## El LLM sobre 10 filas de prueba

In [5]:
# prueba del LLM: 30 filas de entrenamiento como ejemplos y 10 filas de prueba sin respuesta
datos_crudos = pd.read_csv("../data/processed/obt_sales.csv", low_memory=False)
columnas_prompt = ["ITEM_ID", "DEPT_ID", "STORE_ID", "DATE", "WEEKDAY", "SELL_PRICE", "IS_SNAP", "EVENT_NAME_1", "IS_PRE_LAUNCH"]


def describir(fila):
    precio = "sin precio (aun no se vende)" if pd.isna(fila["SELL_PRICE"]) else f"precio {fila['SELL_PRICE']:.2f}"
    evento = fila["EVENT_NAME_1"] if isinstance(fila["EVENT_NAME_1"], str) else "ninguno"
    snap = "si" if fila["IS_SNAP"] else "no"
    return (f"producto {fila['ITEM_ID']}, tienda {fila['STORE_ID']}, fecha {fila['DATE']} ({fila['WEEKDAY']}), "
            f"{precio}, dia SNAP: {snap}, evento: {evento}")


# las filas de los conjuntos se identifican por su posicion en el CSV original
ejemplos = X_ent.sample(30, random_state=42).index
elegidas = X_prueba.sample(10, random_state=42).index

lineas = ["Eres un analista de ventas de una cadena de supermercados. Cada linea describe un producto, una tienda y un dia, "
          "y termina con las unidades vendidas ese dia. Estos son ejemplos:", ""]
for i in ejemplos:
    lineas.append(f"- {describir(datos_crudos.loc[i])} -> unidades vendidas: {datos_crudos.loc[i, 'UNITS']}")
lineas += ["", "Ahora estima las unidades vendidas en estos 10 casos. Responde solo con 10 numeros enteros, uno por linea, en el mismo orden, sin texto:", ""]
for n, i in enumerate(elegidas, 1):
    lineas.append(f"{n}. {describir(datos_crudos.loc[i])}")

with open("prompt_llm.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lineas) + "\n")

Llevé `prompt_llm.txt` a un LLM (Claude), que contestó los 10 números sin ver la columna `UNITS` de las filas de prueba, y los guardé en `respuestas_llm.txt`. Ahora comparo las tres opciones sobre esas mismas 10 filas.

In [6]:
respuestas = np.loadtxt("respuestas_llm.txt")
reales = y_prueba.loc[elegidas].to_numpy()

# la linea base se entrena aparte, solo para tener su columna en la comparacion
linea_base = Pipeline([("preparar", construir_preparacion()), ("modelo", DummyRegressor(strategy="median"))])
linea_base.fit(X_ent, y_ent)

filas = X_prueba.loc[elegidas]
comparacion = pd.DataFrame({
    "real": reales,
    "linea base": linea_base.predict(filas),
    "bosque": entrenados["bosque min_samples_leaf=20"].predict(filas).round(2),
    "red": entrenados["red neuronal (32, 16)"].predict(filas).round(2),
    "llm": respuestas,
}, index=range(1, 11))
print(comparacion)

rmse_10 = {col: round(root_mean_squared_error(reales, comparacion[col]), 4)
           for col in ["linea base", "bosque", "red", "llm"]}
print("RMSE sobre las 10 filas:", rmse_10)

    real  linea base  bosque   red  llm
1      3         0.0    1.10  1.24  1.0
2      0         0.0    1.26  0.92  0.0
3      0         0.0    0.16  0.30  0.0
4      0         0.0    0.71  0.40  0.0
5      1         0.0    0.90  1.00  1.0
6      0         0.0    0.43  0.42  1.0
7      0         0.0    0.39  0.43  0.0
8      0         0.0    0.24  0.14  0.0
9      1         0.0    0.37  0.81  0.0
10     0         0.0    0.29  0.14  0.0
RMSE sobre las 10 filas: {'linea base': 1.0488, 'bosque': 0.8132, 'red': 0.6805, 'llm': 0.7746}


## Tabla de comparación

Con la métrica de prueba completa (39,962 filas) para el bosque y la red, y con las 10 filas para el LLM. El costo del LLM es una estimación: el prompt tiene unos 2,500 tokens de entrada y 10 de salida por cada 10 filas.